In [1]:
!pip install -q yt-dlp pydub librosa soundfile sarvamai datasets huggingface_hub pandas
!apt-get install -qq ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.3/269.3 kB 11.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
path   = "/content/drive/MyDrive/sarvam_assignment"
dataset = f"{path}/dataset/mal"
output = f"{path}/mal_segments"

In [4]:
VIDEOS = {
    "ml_v1": ("https://youtu.be/dvou_xELjJ0", 0),
    "ml_v2": ("https://youtu.be/qXWBnNpuBxA", 4),
    "ml_v3": ("https://youtu.be/WqeOIV3dUos", 0),
}

min_silence_duration = 0.4
silence_threshold_db = -35
min_clip_duration = 5
max_clip_duration = 28
min_snr_db = 15
trim_start=0
trim_end = 6

import os
os.makedirs(dataset, exist_ok=True)
os.makedirs(output, exist_ok=True)

In [5]:
from google.colab import userdata

def _get_secret(name):
    try:
        value = userdata.get(name)
    except Exception:
        value = ""
    if not value:
        raise ValueError(
            f"Secret '{name}' not found or notebook access not granted. "
        )
    return value

SARVAM_API_KEY  = _get_secret("SARVAM_API_KEY1")

In [6]:
import librosa
import numpy as np
import soundfile as sf
import json
import time
import math
import subprocess
import random
import requests
from datetime import datetime
from collections import Counter
from pathlib import Path
import IPython.display as ipd
import pandas as pd

from sarvamai import SarvamAI
sarvam_client = SarvamAI(api_subscription_key=SARVAM_API_KEY)

In [7]:
# Download and load video
def download_video(url, video_id):
    path1 = str(f"{dataset}/{video_id}.wav")
    if os.path.exists(path1):
        print(f"  Already downloaded: {video_id} — skipping.")
        return path1

    print(f"  Downloading {video_id}...")
    out_template = str(path1.replace(".wav", ".%(ext)s"))

    result = subprocess.run([
        "yt-dlp", "-x",
        "--audio-format", "wav",
        "--audio-quality", "0",
        "-o", out_template,
        str(url)
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print(f"  ERROR: {result.stderr[:300]}")
        return None

    print(f"  Saved: {path1}")
    return path1

def load_video(path, sr=16000, start_sec=0, end_sec=0):
    audio, _ = librosa.load(path, sr=sr, mono=True)

    start = int(start_sec * sr)
    end = int(len(audio) - (end_sec * sr))
    start = max(0, start)
    end = max(start + 1, end)
    trimmed = audio[start:end]
    duration_mins = (len(trimmed) / sr) / 60
    print(f"Trimmed {start_sec}s intro and {end_sec}s outro, {duration_mins:.1f} min remaining")
    return trimmed, sr

In [8]:
def split_audio(audio, sr):
    hop_length   = 512
    frame_length = 2048
    rms = librosa.feature.rms(y=audio,
                               frame_length=frame_length,
                               hop_length=hop_length)[0]
    db  = librosa.amplitude_to_db(rms, ref=np.max)

    silence_frames     = db < silence_threshold_db
    min_silence_frames = int(min_silence_duration * sr / hop_length)

    boundaries, in_silence, silence_start = [], False, 0
    for i, is_silent in enumerate(silence_frames):
        if is_silent and not in_silence:
            silence_start = i
            in_silence    = True
        elif not is_silent and in_silence:
            if (i - silence_start) >= min_silence_frames:
                boundaries.append(((silence_start + i) // 2) * hop_length)
            in_silence = False

    segments, prev = [], 0
    for cut in boundaries + [len(audio)]:
        chunk    = audio[prev:cut]
        duration = len(chunk) / sr
        if min_clip_duration <= duration <= max_clip_duration:
            segments.append((chunk, duration))
        elif duration > max_clip_duration:
            mid = len(chunk) // 2
            for half in [chunk[:mid], chunk[mid:]]:
                if len(half) / sr >= min_clip_duration:
                    segments.append((half, len(half) / sr))
        prev = cut
    return segments


def compute_snr(segment):
    rms         = np.sqrt(np.mean(segment ** 2))
    noise_floor = np.percentile(np.abs(segment), 10)
    return 20 * np.log10(rms / (noise_floor + 1e-8))

def add_fades(segment, sr, fade_ms=50):
    n   = int(fade_ms * sr / 1000)
    seg = segment.copy()
    seg[:n]  *= np.linspace(0, 1, n)
    seg[-n:] *= np.linspace(1, 0, n)
    return seg

In [9]:
def process_video(video_id, url, output, lang_code,
                  start_sec, end_sec):

    print(f"  {video_id} | {lang_code}")

    raw_path = download_video(url, video_id)
    if raw_path is None:
        return [], [{"id": video_id, "reason": "download_failed", "duration": 0}]

    audio, sr = load_video(raw_path,
                               start_sec=start_sec,
                               end_sec=end_sec)
    segments = split_audio(audio, sr)
    print(f"  Candidate segments: {len(segments)}")

    accepted, rejected = [], []
    for i, (segment, duration) in enumerate(segments):
        seg_id = f"{video_id}_seg{i:03d}"
        snr    = compute_snr(segment)
        if snr < min_snr_db:
            rejected.append({"id": seg_id, "duration": round(duration, 2),
                             "reason": f"low_snr:{snr:.1f}dB"})
            continue
        segment  = add_fades(segment, sr)
        filepath = os.path.join(output, f"{seg_id}.wav")
        sf.write(filepath, segment, sr)
        accepted.append({
            "id": seg_id,
            "file": filepath,
            "lang_code": lang_code,
            "duration_sec": round(duration, 2),
            "snr_db": round(snr, 2),
            "source_url": url
        })

    total_min = sum(s["duration_sec"] for s in accepted) / 60
    print(f"  Accepted : {len(accepted)} segments ({total_min:.1f} min)")
    print(f"  Rejected : {len(rejected)} segments")
    return accepted, rejected

In [10]:
all_accepted = []
all_rejected = []

for video_id, url_data in VIDEOS.items():
    url = url_data
    start_sec = trim_start

    if isinstance(url_data, (tuple, list)):
        url = url_data[0]
        start_sec = url_data[1]

    acc, rej = process_video(
        video_id=video_id,
        url=url,
        output=output,
        lang_code= "ml-IN",
        start_sec=start_sec,
        end_sec=trim_end
    )

    all_accepted.extend(acc)
    all_rejected.extend(rej)

total_min = sum(s["duration_sec"] for s in all_accepted) / 60

print(f"Accepted : {len(all_accepted):>4} segments | {total_min:.1f} min")
print(f"Rejected : {len(all_rejected):>4} segments")

  ml_v1 | ml-IN
  Saved: /content/drive/MyDrive/sarvam_assignment/dataset/mal/ml_v1.wav
Trimmed 0s intro and 6s outro, 14.9 min remaining
  Candidate segments: 65
  Accepted : 65 segments (10.7 min)
  Rejected : 0 segments
  ml_v2 | ml-IN
  Saved: /content/drive/MyDrive/sarvam_assignment/dataset/mal/ml_v2.wav
Trimmed 4s intro and 6s outro, 4.1 min remaining
  Candidate segments: 17
  Accepted : 17 segments (3.2 min)
  Rejected : 0 segments
  ml_v3 | ml-IN
  Saved: /content/drive/MyDrive/sarvam_assignment/dataset/mal/ml_v3.wav
Trimmed 0s intro and 6s outro, 11.0 min remaining
  Candidate segments: 2
  Accepted : 2 segments (11.0 min)
  Rejected : 0 segments
Accepted :   84 segments | 24.9 min
Rejected :    0 segments


In [11]:
if all_accepted:
    pick = all_accepted[0]
    print(f"File     : {pick['file']}")
    print(f"Duration : {pick['duration_sec']}s | SNR: {pick['snr_db']}dB")
    ipd.display(ipd.Audio(pick["file"]))
else:
    print(f"No Malayalam segments found.")

File     : /content/drive/MyDrive/sarvam_assignment/mal_segments/ml_v1_seg000.wav
Duration : 7.07s | SNR: 34.400001525878906dB


In [12]:
def transcribe_sarvam(audio_path, language_code):
    url     = "https://api.sarvam.ai/speech-to-text"
    headers = {"api-subscription-key": SARVAM_API_KEY}

    with open(audio_path, "rb") as f:
        resp = requests.post(
            url, headers=headers,
            files={"file": (os.path.basename(audio_path), f, "audio/wav")},
            data={"model":             "saaras:v3",
                  "mode":              "transcribe",
                  "language_code":     language_code,
                  "with_timestamps":   False,
                  "with_disfluencies": False}
        )
    if resp.status_code == 200:
        return resp.json().get("transcript", "").strip()
    print(f"  ASR error {resp.status_code}: {resp.text[:200]}")
    return None

In [13]:
failed_asr = []
total      = len(all_accepted)

for i, seg in enumerate(all_accepted):
    print(f"  [{i+1}/{total}] {seg['id']}...", end=" ")
    transcript = transcribe_sarvam(seg["file"], "ml-IN")
    if transcript and len(transcript.strip()) > 3:
        seg["transcript"] = transcript
        print(f"OK ({len(transcript)} chars)")
    else:
        seg["transcript"] = None
        failed_asr.append(seg["id"])
        print("EMPTY")
    time.sleep(0.5)

before       = len(all_accepted)
all_accepted = [s for s in all_accepted if s.get("transcript")]
print(f"\nPassed  : {len(all_accepted)}")
print(f"Filtered: {before - len(all_accepted)} (empty transcript)")
if failed_asr:
    print(f"Failed IDs: {failed_asr}")

  [1/84] ml_v1_seg000... OK (60 chars)
  [2/84] ml_v1_seg001... OK (82 chars)
  [3/84] ml_v1_seg002... OK (183 chars)
  [4/84] ml_v1_seg003... OK (84 chars)
  [5/84] ml_v1_seg004... OK (42 chars)
  [6/84] ml_v1_seg005... OK (93 chars)
  [7/84] ml_v1_seg006... OK (163 chars)
  [8/84] ml_v1_seg007... OK (189 chars)
  [9/84] ml_v1_seg008... OK (129 chars)
  [10/84] ml_v1_seg009... OK (94 chars)
  [11/84] ml_v1_seg010... OK (75 chars)
  [12/84] ml_v1_seg011... OK (74 chars)
  [13/84] ml_v1_seg012... OK (53 chars)
  [14/84] ml_v1_seg013... OK (154 chars)
  [15/84] ml_v1_seg014... OK (77 chars)
  [16/84] ml_v1_seg015... OK (45 chars)
  [17/84] ml_v1_seg016... OK (51 chars)
  [18/84] ml_v1_seg017... OK (69 chars)
  [19/84] ml_v1_seg018... OK (267 chars)
  [20/84] ml_v1_seg019... OK (85 chars)
  [21/84] ml_v1_seg020... OK (103 chars)
  [22/84] ml_v1_seg021... OK (45 chars)
  [23/84] ml_v1_seg022... OK (132 chars)
  [24/84] ml_v1_seg023... OK (86 chars)
  [25/84] ml_v1_seg024... OK (85 chars)
 

In [14]:
def tag_emotion_video(transcript, language_code):
    prompt = f"""You are a TTS dataset curator. Analyze this Malayalam speech segment and assign tags.

Transcript: {transcript}

Respond ONLY with valid JSON, no markdown, no explanation:
{{"emotion": "<happy|sad|excited|angry|neutral|formal|whisper|frustrated>", "style": "<conversational|narrative|formal|instructional>", "confidence": "<high|medium|low>", "register": "<formal|colloquial|code-mixed>"}}"""

    try:
        response = sarvam_client.chat.completions(
            model="sarvam-105b",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300,
            temperature=0.1,
            reasoning_effort=None
        )
        content = response.choices[0].message.content
        if content is None:
            print("  LLM returned None content")
            return None
        content = content.replace("```json", "").replace("```", "").strip()
        return json.loads(content)
    except json.JSONDecodeError as e:
        print(f"  JSON parse error: {e} | Raw: {content[:100]}")
        return None
    except Exception as e:
        print(f"  Sarvam LLM error: {e}")
        return None


def tag_emotion(transcript, language_code):
    result = tag_emotion_video(transcript, language_code)
    return result or {"emotion": "neutral", "style": "conversational",
                      "confidence": "low", "register": "colloquial"}

In [15]:
# Run tagging
total = len(all_accepted)
for i, seg in enumerate(all_accepted):

    if not seg.get("transcript"):
        print(f"  [{i+1}/{total}] {seg['id']}... SKIP (no transcript)")
        seg.update({"emotion": "neutral", "style": "conversational",
                    "confidence": "low", "register": "colloquial"})
        continue

    print(f"  [{i+1}/{total}] {seg['id']}...", end=" ")
    tags = tag_emotion(seg["transcript"], "ml-IN")
    seg.update(tags)
    print(f"{tags['emotion']} / {tags['style']} / conf:{tags['confidence']}")
    time.sleep(0.2)

emotions   = Counter(s.get("emotion",   "?") for s in all_accepted)
confidence = Counter(s.get("confidence","?") for s in all_accepted)
registers  = Counter(s.get("register",  "?") for s in all_accepted)

print(f"\nEmotion    : {dict(emotions)}")
print(f"Confidence : {dict(confidence)}")
print(f"Register   : {dict(registers)}")

  [1/82] ml_v1_seg000... neutral / formal / conf:high
  [2/82] ml_v1_seg001... neutral / formal / conf:high
  [3/82] ml_v1_seg002... excited / narrative / conf:high
  [4/82] ml_v1_seg003... neutral / narrative / conf:high
  [5/82] ml_v1_seg004... neutral / conversational / conf:high
  [6/82] ml_v1_seg005... neutral / narrative / conf:high
  [7/82] ml_v1_seg006... neutral / narrative / conf:high
  [8/82] ml_v1_seg007... neutral / narrative / conf:high
  [9/82] ml_v1_seg008... neutral / narrative / conf:high
  [10/82] ml_v1_seg009... neutral / narrative / conf:high
  [11/82] ml_v1_seg010... neutral / narrative / conf:high
  [12/82] ml_v1_seg011... neutral / narrative / conf:high
  [13/82] ml_v1_seg012... neutral / narrative / conf:high
  [14/82] ml_v1_seg013... neutral / narrative / conf:high
  [15/82] ml_v1_seg014... neutral / narrative / conf:high
  [16/82] ml_v1_seg015... neutral / narrative / conf:high
  [17/82] ml_v1_seg016... neutral / narrative / conf:high
  [18/82] ml_v1_seg017..

In [16]:
SOURCE_META = {
    "ml_v1": "Malayalam Video 1",
    "ml_v2": "Malayalam Video 2",
    "ml_v3": "Malayalam Video 3"
}

final_metadata = []
for idx, seg in enumerate(all_accepted):
    video_id = "_".join(seg["id"].split("_")[:2])
    entry = {
      "id":             f"mal_{idx+1:04d}",
      "file_path":      seg["file"],
      "language":       "ml-IN",
      "language_name":  "Malayalam",
      "speaker_id":     "spk_01",
      "duration_sec":   float(seg["duration_sec"]),
      "transcript":     seg["transcript"],
      "emotion":        seg.get("emotion",    "neutral"),
      "style":          seg.get("style",      "conversational"),
      "register":       seg.get("register",   "colloquial"),
      "tag_confidence": seg.get("confidence", "low"),
      "snr_db":         float(seg["snr_db"]),
      "source_url":     seg["source_url"],
      "source_channel": SOURCE_META.get(video_id, ""),
      "curated_at":     datetime.now().isoformat()
    }
    final_metadata.append(entry)

# Save JSONL
with open(f"{path}/metadata.jsonl", "w", encoding="utf-8") as f:
    for e in final_metadata:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")

# Save rejection log
with open(f"{path}/rejected_log.jsonl", "w", encoding="utf-8") as f:
    for r in all_rejected:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

total_dur = sum(e["duration_sec"] for e in final_metadata) / 60
rej_rate  = len(all_rejected) / max(1, len(all_rejected) + len(final_metadata)) * 100


print(f"Malayalam")
print(f"  Segments  : {len(final_metadata):>4} | {total_dur:.1f} min")
print(f"  Rejected  : {len(all_rejected):>4} segments ({rej_rate:.1f}% rejection rate)")

Malayalam
  Segments  :   82 | 13.9 min
  Rejected  :    0 segments (0.0% rejection rate)


In [17]:
sample = random.sample(final_metadata, min(12, len(final_metadata)))
print(f"{'ID':<20} {'Dur':>5} {'SNR':>5} {'Emotion':<12} {'Conf':<8} Transcript[:70]")
print("-" * 100)
for e in sample:
    print(f"{e['id']:<20} {e['duration_sec']:>4.0f}s "
          f"{e['snr_db']:>4.0f}dB {e['emotion']:<12} {e['tag_confidence']:<8} "
          f"{e['transcript'][:70]}")

pick = random.choice(final_metadata)
print(f"ID         : {pick['id']}")
print(f"Transcript : {pick['transcript']}")
print(f"Emotion    : {pick['emotion']} | Register: {pick['register']}")
ipd.display(ipd.Audio(pick["file_path"]))

ID                     Dur   SNR Emotion      Conf     Transcript[:70]
----------------------------------------------------------------------------------------------------
mal_0048                6s   37dB neutral      high     പറഞ്ഞു തുടങ്ങിയത് ശ്രീ എം പി വീരേന്ദ്രകുമാർ എന്നെപ്പോലെ വളരെ
mal_0056               15s   31dB neutral      high     ശ്രീ എം.ബി.ബി. രചേന്ദ്രകുമാർ ഭയന്നതുപോലെ ദുരന്തങ്ങൾ ആവർത്തിക്കുന്നത് ന
mal_0038                5s   32dB neutral      high     ഗാന്ധിജിയുടെ ചോദ്യത്തിന് അയ്യങ്കാളി പറഞ്ഞ മറുപടി ഓർമ്മിക്കേണ്ടതാണ്.
mal_0003               18s   30dB excited      high     ഈ അക്ഷരോത്സവം കേരളത്തിലെ ഒരുപക്ഷേ സ്ത്രീകൾ നേതൃത്വം നൽകുന്നതും സംഘടിപ്
mal_0050                6s   48dB neutral      high     ആദ്യ വായനയിൽ അന്നത്തെ പ്രായക്കുറവും ജ്ഞാനക്കുറവും ബോധക്കുറവും ഒക്കെ മൂ
mal_0005                5s   34dB neutral      high     സദസ്യർക്കിടയിൽ കസേര കിട്ടാതെ നിന്നുകൊണ്ട്.
mal_0007               16s   34dB neutral      high     നമുക്ക് കിട്ടുന്ന വലിയ പുരസ്കാരങ്ങൾ അല്ല ഓർമ്മയിൽ ത

In [18]:
def categorize_rejection(reason):
    r = reason.lower()
    if "snr"          in r: return "background_noise"
    if "multi_speaker" in r: return "multi_speaker"
    if "download"      in r: return "download_failed"
    if "too_short"     in r: return "too_short"
    return "other"

cats = Counter(categorize_rejection(r.get("reason", "")) for r in all_rejected)
print(f"Total rejected: {len(all_rejected)}")
print(f"Rejection rate: {rej_rate:.1f}%\n")
if all_rejected:
  print("Breakdown:")
  for cat, count in cats.most_common():
      print(f"  {cat:<25} {count:>4}")

Total rejected: 0
Rejection rate: 0.0%



In [19]:
NEW_VIDEOS = {
    "ml_v4": ("https://youtu.be/oJltochNzR8", 0),
    "ml_v5": ("https://youtu.be/9dt-M8kkbbU", 3),
    "ml_v6": ("https://youtu.be/kAv3NPPvvAc", 0),
}

for video_id, url_data in NEW_VIDEOS.items():
    url       = url_data[0]
    start_sec = url_data[1]
    acc, rej  = process_video(
        video_id=video_id,
        url=url,
        output=output,
        lang_code="ml-IN",
        start_sec=start_sec,
        end_sec=trim_end
    )
    all_accepted.extend(acc)
    all_rejected.extend(rej)

total_min = sum(s["duration_sec"] for s in all_accepted) / 60
print(f"Total: {len(all_accepted)} segments | {total_min:.1f} min")

  ml_v4 | ml-IN
  Saved: /content/drive/MyDrive/sarvam_assignment/dataset/mal/ml_v4.wav
Trimmed 0s intro and 6s outro, 4.4 min remaining
  Candidate segments: 15
  Accepted : 15 segments (3.8 min)
  Rejected : 0 segments
  ml_v5 | ml-IN
  Saved: /content/drive/MyDrive/sarvam_assignment/dataset/mal/ml_v5.wav
Trimmed 3s intro and 6s outro, 5.0 min remaining
  Candidate segments: 21
  Accepted : 21 segments (3.7 min)
  Rejected : 0 segments
  ml_v6 | ml-IN
  Saved: /content/drive/MyDrive/sarvam_assignment/dataset/mal/ml_v6.wav
Trimmed 0s intro and 6s outro, 11.2 min remaining
  Candidate segments: 44
  Accepted : 44 segments (9.6 min)
  Rejected : 0 segments
Total: 162 segments | 31.0 min


In [20]:
to_transcribe = [s for s in all_accepted if not s.get("transcript")]
print(f"Segments to transcribe: {len(to_transcribe)}")

failed_asr = []
total = len(to_transcribe)

for i, seg in enumerate(to_transcribe):
    print(f"  [{i+1}/{total}] {seg['id']}...", end=" ")
    transcript = transcribe_sarvam(seg["file"], "ml-IN")
    if transcript and len(transcript.strip()) > 3:
        seg["transcript"] = transcript
        print(f"OK ({len(transcript)} chars)")
    else:
        seg["transcript"] = None
        failed_asr.append(seg["id"])
        print("EMPTY")
    time.sleep(0.5)

before       = len(all_accepted)
all_accepted = [s for s in all_accepted if s.get("transcript")]
print(f"\nPassed  : {len(all_accepted)}")
print(f"Filtered: {before - len(all_accepted)}")

Segments to transcribe: 80
  [1/80] ml_v4_seg000... OK (324 chars)
  [2/80] ml_v4_seg001... OK (270 chars)
  [3/80] ml_v4_seg002... OK (278 chars)
  [4/80] ml_v4_seg003... OK (160 chars)
  [5/80] ml_v4_seg004... OK (224 chars)
  [6/80] ml_v4_seg005... OK (155 chars)
  [7/80] ml_v4_seg006... OK (125 chars)
  [8/80] ml_v4_seg007... OK (127 chars)
  [9/80] ml_v4_seg008... OK (25 chars)
  [10/80] ml_v4_seg009... OK (236 chars)
  [11/80] ml_v4_seg010... OK (197 chars)
  [12/80] ml_v4_seg011... OK (102 chars)
  [13/80] ml_v4_seg012... OK (153 chars)
  [14/80] ml_v4_seg013... OK (164 chars)
  [15/80] ml_v4_seg014... OK (53 chars)
  [16/80] ml_v5_seg000... OK (169 chars)
  [17/80] ml_v5_seg001... OK (75 chars)
  [18/80] ml_v5_seg002... OK (64 chars)
  [19/80] ml_v5_seg003... OK (186 chars)
  [20/80] ml_v5_seg004... OK (113 chars)
  [21/80] ml_v5_seg005... OK (58 chars)
  [22/80] ml_v5_seg006... OK (129 chars)
  [23/80] ml_v5_seg007... OK (60 chars)
  [24/80] ml_v5_seg008... OK (37 chars)
  [25

In [22]:
to_tag = [s for s in all_accepted if not s.get("emotion")]

total = len(to_tag)
for i, seg in enumerate(to_tag):
    if not seg.get("transcript"):
        continue
    print(f"  [{i+1}/{total}] {seg['id']}...", end=" ")
    tags = tag_emotion(seg["transcript"], "ml-IN")
    seg.update(tags)
    print(f"{tags['emotion']} / {tags['style']} / conf:{tags['confidence']}")
    time.sleep(0.2)

  [1/78] ml_v4_seg000... frustrated / conversational / conf:high
  [2/78] ml_v4_seg001... frustrated / conversational / conf:high
  [3/78] ml_v4_seg002... sad / narrative / conf:high
  [4/78] ml_v4_seg003... excited / conversational / conf:high
  [5/78] ml_v4_seg004... neutral / conversational / conf:high
  [6/78] ml_v4_seg005... happy / conversational / conf:high
  [7/78] ml_v4_seg006... frustrated / conversational / conf:high
  [8/78] ml_v4_seg007... frustrated / conversational / conf:high
  [9/78] ml_v4_seg008... neutral / conversational / conf:high
  [10/78] ml_v4_seg009... frustrated / conversational / conf:high
  [11/78] ml_v4_seg010... frustrated / conversational / conf:high
  [12/78] ml_v4_seg011... angry / conversational / conf:high
  [13/78] ml_v4_seg012... excited / conversational / conf:high
  [14/78] ml_v4_seg013... sad / conversational / conf:high
  [15/78] ml_v4_seg014... neutral / conversational / conf:medium
  [16/78] ml_v5_seg000... neutral / narrative / conf:high
  [

In [23]:
emotions   = Counter(s.get("emotion",   "?") for s in all_accepted)
confidence = Counter(s.get("confidence","?") for s in all_accepted)
registers  = Counter(s.get("register",  "?") for s in all_accepted)

print(f"\nEmotion    : {dict(emotions)}")
print(f"Confidence : {dict(confidence)}")
print(f"Register   : {dict(registers)}")


Emotion    : {'neutral': 120, 'excited': 8, 'frustrated': 17, 'angry': 7, 'sad': 6, 'happy': 2}
Confidence : {'high': 156, 'medium': 4}
Register   : {'formal': 70, 'colloquial': 88, 'code-mixed': 2}


In [24]:
SOURCE_META = {
    "ml_v1": "Malayalam Video 1",
    "ml_v2": "Malayalam Video 2",
    "ml_v3": "Malayalam Video 3"
}

final_metadata = []
for idx, seg in enumerate(all_accepted):
    video_id = "_".join(seg["id"].split("_")[:2])
    entry = {
      "id":             f"mal_{idx+1:04d}",
      "file_path":      seg["file"],
      "language":       "ml-IN",
      "language_name":  "Malayalam",
      "speaker_id":     "spk_01",
      "duration_sec":   float(seg["duration_sec"]),
      "transcript":     seg["transcript"],
      "emotion":        seg.get("emotion",    "neutral"),
      "style":          seg.get("style",      "conversational"),
      "register":       seg.get("register",   "colloquial"),
      "tag_confidence": seg.get("confidence", "low"),
      "snr_db":         float(seg["snr_db"]),
      "source_url":     seg["source_url"],
      "source_channel": SOURCE_META.get(video_id, ""),
      "curated_at":     datetime.now().isoformat()
    }
    final_metadata.append(entry)

# Save JSONL
with open(f"{path}/metadata.jsonl", "w", encoding="utf-8") as f:
    for e in final_metadata:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")

# Save rejection log
with open(f"{path}/rejected_log.jsonl", "w", encoding="utf-8") as f:
    for r in all_rejected:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

total_dur = sum(e["duration_sec"] for e in final_metadata) / 60
rej_rate  = len(all_rejected) / max(1, len(all_rejected) + len(final_metadata)) * 100


print(f"Malayalam")
print(f"  Segments  : {len(final_metadata):>4} | {total_dur:.1f} min")
print(f"  Rejected  : {len(all_rejected):>4} segments ({rej_rate:.1f}% rejection rate)")

Malayalam
  Segments  :  160 | 29.8 min
  Rejected  :    0 segments (0.0% rejection rate)
